# 🎤 AI Singer × HeartMuLa
## 가사 + 태그 → AI 작곡 → 완성곡 생성

**HeartMuLa 3B (4-bit 양자화)** 를 사용하여 Colab 무료 T4 GPU에서 음악을 생성합니다.

### 사용법
1. **Step 1**: 환경 설치 (최초 1회, ~5분)
2. **Step 2**: 모델 다운로드 (최초 1회, ~10분)
3. **Step 3**: 가사 & 태그 입력
4. **Step 4**: 곡 생성! 🎵

---
*Built by KK & Chloe 🦞*

## Step 0: GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("❌ GPU를 사용할 수 없습니다. Runtime > Change runtime type > T4 GPU 를 선택하세요.")

## Step 1: 환경 설치 (~5분)

In [ ]:
# HeartMuLa 라이브러리 설치
!git clone https://github.com/HeartMuLa/heartlib.git
%cd heartlib
!pip install -e . -q

# 4-bit 양자화를 위한 추가 패키지
!pip install bitsandbytes accelerate -q

print("\n✅ 환경 설치 완료!")

## Step 2: 모델 다운로드 (~10분)

4-bit 양자화 모델 + HeartCodec + 토크나이저를 다운로드합니다.

In [ ]:
import os
os.makedirs('./ckpt/HeartMuLa-oss-3B', exist_ok=True)
os.makedirs('./ckpt/HeartCodec-oss', exist_ok=True)

# 4-bit 양자화 모델 다운로드
print("📥 HeartMuLa 3B (4-bit) 다운로드 중...")
!huggingface-cli download PavonicAI/HeartMuLa-3B-4bit --local-dir ./ckpt/HeartMuLa-oss-3B --quiet

# HeartCodec 다운로드
print("\n📥 HeartCodec 다운로드 중...")
!huggingface-cli download HeartMuLa/HeartCodec-oss-20260123 --local-dir ./ckpt/HeartCodec-oss --quiet

# 토크나이저 + 설정 다운로드
print("\n📥 토크나이저 & 설정 다운로드 중...")
!huggingface-cli download HeartMuLa/HeartMuLaGen --local-dir ./ckpt --include "tokenizer.json" "gen_config.json" --quiet

print("\n✅ 모든 모델 다운로드 완료!")
print("\n📂 체크포인트 구조:")
!find ./ckpt -maxdepth 2 -type f | head -20

In [ ]:
# VRAM 사용량 확인
!nvidia-smi | grep MiB

## Step 3: 가사 & 태그 입력 ✍️

아래 셀에서 가사와 태그를 수정하세요.

In [ ]:
# ═══════════════════════════════════════
# 🎵 여기서 가사와 태그를 수정하세요!
# ═══════════════════════════════════════

LYRICS = """
[Verse]
새벽 세시 모니터 불빛 아래
코드 한 줄에 세상을 담아
에러 메시지 수백 개를 넘어서
드디어 만든 나만의 세계

[Prechorus]
멈추지 마 이 순간을
꿈이 현실이 되는 그 순간을

[Chorus]
코드로 만드는 내 우주
0과 1 사이의 마법
아무도 가지 않은 길 위에서
나는 오늘도 달려간다

[Verse]
실패해도 괜찮아 다시 시작해
버그는 성장의 다른 이름이야
커피 한 잔에 용기를 채우고
키보드 위에 꿈을 그려가

[Chorus]
코드로 만드는 내 우주
0과 1 사이의 마법
아무도 가지 않은 길 위에서
나는 멈추지 않는 개발자

[Outro]
끝없는 도전 그 끝에서
빛나는 내일을 만나리
"""

TAGS = "energetic,pop,rock,drums,electric guitar,synthesizer,hopeful,Korean"

# 곡 길이 (밀리초, 최대 240000 = 4분)
MAX_LENGTH_MS = 180000  # 3분

# CFG Scale (2.0=팝/발라드, 3.0=록/라틴, 4.0+=일렉트로닉)
CFG_SCALE = 2.5

# ═══════════════════════════════════════

# 파일로 저장
with open('./assets/lyrics.txt', 'w') as f:
    f.write(LYRICS.strip())
with open('./assets/tags.txt', 'w') as f:
    f.write(TAGS)

print("✅ 가사 & 태그 저장 완료!")
print(f"\n🎵 태그: {TAGS}")
print(f"⏱️ 최대 길이: {MAX_LENGTH_MS/1000:.0f}초")
print(f"🎛️ CFG Scale: {CFG_SCALE}")
print(f"\n📝 가사 미리보기:")
print(LYRICS[:300] + "...")

## Step 4: 🎶 곡 생성!

4-bit 모델로 생성합니다. T4 GPU 기준 RTF ≈ 1.0 (3분곡 → 약 3분 소요)

In [ ]:
import time
start = time.time()

print("🎤 HeartMuLa 음악 생성 시작!")
print(f"   모델: HeartMuLa 3B (4-bit NF4)")
print(f"   길이: {MAX_LENGTH_MS/1000:.0f}초")
print(f"   CFG: {CFG_SCALE}")
print("   ⏳ 생성 중... (T4 기준 곡 길이와 비슷한 시간 소요)")
print()

!python ./examples/run_music_generation.py \
    --model_path=./ckpt \
    --version="3B" \
    --lyrics=./assets/lyrics.txt \
    --tags=./assets/tags.txt \
    --save_path=./output.mp3 \
    --max_audio_length_ms={MAX_LENGTH_MS} \
    --cfg_scale={CFG_SCALE} \
    --lazy_load=true

elapsed = time.time() - start
print(f"\n✅ 생성 완료! ({elapsed:.0f}초 소요)")

## Step 5: 🎧 결과 재생 & 다운로드

In [ ]:
import os
from IPython.display import Audio, display, HTML
from google.colab import files

output_path = './output.mp3'

if os.path.exists(output_path):
    size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f"🎵 생성된 곡: {output_path} ({size_mb:.1f} MB)")
    print()

    # 재생
    display(Audio(output_path))

    # 다운로드 버튼
    print("\n⬇️ 아래 버튼으로 다운로드:")
    files.download(output_path)
else:
    print("❌ 출력 파일이 없습니다. Step 4의 로그를 확인하세요.")
    print("\n💡 Tip: VRAM 부족이면 lazy_load=true 옵션을 확인하세요.")

## 🎛️ 태그 가이드

| 카테고리 | 예시 태그 |
|---------|----------|
| **장르** | pop, rock, ballad, R&B, electronic, hip-hop, jazz |
| **악기** | piano, electric guitar, drums, synthesizer, strings, acoustic guitar |
| **분위기** | happy, sad, energetic, calm, romantic, epic, hopeful |
| **언어** | Korean, English, Japanese, Chinese, Spanish |
| **기타** | wedding, cafe, driving, meditation, powerful |

### CFG Scale 가이드
| CFG | 추천 장르 |
|-----|----------|
| 2.0 | 팝, 발라드, 감성곡 |
| 2.5 | 팝록, K-pop |
| 3.0 | 록, 라틴, 업템포 |
| 4.0+ | 일렉트로닉, 댄스 (아티팩트 주의) |

## 📊 VRAM 사용량 확인

In [ ]:
!nvidia-smi